<a href="https://www.kaggle.com/code/avikdas567/hybrid-tf-idf-rules-for-sdrf-metadata-extraction?scriptVersionId=306717385" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import json
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE_PATH = "/kaggle/input/competitions/harmonizing-the-data-of-your-data"

TRAIN_SDRF_PATH = f"{BASE_PATH}/Training_SDRFs"
TRAIN_GPT_PATH = f"{BASE_PATH}/Training_GPT_Extract"
TEST_TEXT_PATH = f"{BASE_PATH}/Test PubText/Test PubText"
SAMPLE_SUB = f"{BASE_PATH}/SampleSubmission.csv"

sample_sub = pd.read_csv(SAMPLE_SUB)
columns = sample_sub.columns.tolist()

# TEXT

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text

def load_json_text(path):
    with open(path, 'r') as f:
        data = json.load(f)

    texts = []

    def extract(obj):
        if isinstance(obj, dict):
            for v in obj.values():
                extract(v)
        elif isinstance(obj, list):
            for v in obj:
                extract(v)
        elif isinstance(obj, str):
            texts.append(obj)

    extract(data)
    return clean_text(" ".join(texts))

# BUILD KB

print("Building knowledge base...")

knowledge = defaultdict(set)

for root, _, files in os.walk(TRAIN_SDRF_PATH):
    for file in files:
        if file.endswith((".tsv", ".txt", ".csv")):
            try:
                df = pd.read_csv(os.path.join(root, file), sep="\t")
                for col in df.columns:
                    if col != "PXD":
                        for v in df[col].dropna().astype(str):
                            knowledge[col].add(v.lower().strip())
            except:
                pass

for root, _, files in os.walk(TRAIN_GPT_PATH):
    for file in files:
        if file.endswith(".csv"):
            try:
                df = pd.read_csv(os.path.join(root, file))
                for col in df.columns:
                    if col != "PXD":
                        for v in df[col].dropna().astype(str):
                            knowledge[col].add(v.lower().strip())
            except:
                pass

print(f"Knowledge columns: {len(knowledge)}")

# TF-IDF ENSEMBLE

vectorizers_word = {}
vectorizers_char = {}
value_vectors_word = {}
value_vectors_char = {}

for col, values in knowledge.items():
    values = list(values)
    if len(values) < 5:
        continue

    vec_w = TfidfVectorizer(ngram_range=(1,2), max_features=2000)
    X_w = vec_w.fit_transform(values)

    vec_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=2000)
    X_c = vec_c.fit_transform(values)

    vectorizers_word[col] = vec_w
    value_vectors_word[col] = (values, X_w)

    vectorizers_char[col] = vec_c
    value_vectors_char[col] = (values, X_c)

# MATCHING

def extract_smart(text):

    found = defaultdict(set)

    for col in vectorizers_word:

        values_w, X_w = value_vectors_word[col]
        values_c, X_c = value_vectors_char[col]

        sims_w = cosine_similarity(vectorizers_word[col].transform([text]), X_w)[0]
        sims_c = cosine_similarity(vectorizers_char[col].transform([text]), X_c)[0]

        sims = sims_w + sims_c

        top_idx = np.argsort(sims)[-7:]

        for i in top_idx:
            found[col].add(values_w[i])

    return found

# RULES

def rule_boost(text):

    out = defaultdict(set)

    out["Characteristics[Organism]"].add("homo sapiens")

    if "mouse" in text:
        out["Characteristics[Organism]"].add("mus musculus")

    if "trypsin" in text:
        out["Characteristics[CleavageAgent]"].add("trypsin")

    if "orbitrap" in text:
        out["Comment[Instrument]"].add("orbitrap")

    if "mass spectrometry" in text:
        out["Comment[Instrument]"].add("mass spectrometer")

    if "tmt" in text:
        out["Characteristics[Label]"].add("tmt")

    if "silac" in text:
        out["Characteristics[Label]"].add("silac")

    if "cancer" in text:
        out["Characteristics[Disease]"].add("cancer")

    return out

# PROPAGATION

def propagate(row):

    if row.get("Characteristics[Disease]", ""):
        row["FactorValue[Disease]"] = row["Characteristics[Disease]"]

    if row.get("Characteristics[Label]", ""):
        row["FactorValue[Label]"] = row["Characteristics[Label]"]

    if row.get("Characteristics[Organism]", ""):
        row["Characteristics[MaterialType]"] = "tissue"

    if row.get("Characteristics[CleavageAgent]", ""):
        row["FactorValue[CleavageAgent]"] = row["Characteristics[CleavageAgent]"]

    return row

# PROCESS

print("Processing test set...")

predictions = []

test_files = [f for f in os.listdir(TEST_TEXT_PATH) if f.endswith(".json")]

for file in tqdm(test_files):

    pxd = file.split("_")[0]
    text = load_json_text(os.path.join(TEST_TEXT_PATH, file))

    smart = extract_smart(text)
    rule = rule_boost(text)

    combined = defaultdict(set)

    for k in smart:
        combined[k].update(smart[k])

    for k in rule:
        combined[k].update(rule[k])

    row = {col: "" for col in columns}
    row["PXD"] = pxd

    for col in combined:
        if col in row:
            vals = list(set(combined[col]))
            row[col] = "|".join(vals[:3])

    defaults = {
        "Characteristics[Organism]": "homo sapiens",
        "Comment[Instrument]": "mass spectrometer",
        "Characteristics[CleavageAgent]": "trypsin",
        "Characteristics[Label]": "label free",
        "Characteristics[MaterialType]": "tissue",
        "Characteristics[Disease]": "normal",
        "Characteristics[CellType]": "cell",
        "Characteristics[Sex]": "unknown",
        "Characteristics[Temperature]": "37"
    }

    for col, val in defaults.items():
        if col in row and row[col] == "":
            row[col] = val

    row = propagate(row)

    predictions.append(row)

# SAVE

submission = pd.DataFrame(predictions)

for col in columns:
    if col not in submission.columns:
        submission[col] = ""

submission = submission[columns]

submission = submission.astype(str)

submission = submission.replace(
    ["nan", "None", "NaN", "NULL", "<NA>"], ""
)

submission = submission.apply(lambda col: col.astype(str).str.strip())

submission = submission.fillna("")

print("Nulls:", submission.isnull().sum().sum())
assert submission.isnull().sum().sum() == 0

submission.to_csv("submission.csv", index=False)

print("'submission.csv' created.")
display(submission.head())

Building knowledge base...
Knowledge columns: 99
Processing test set...


100%|██████████| 16/16 [03:20<00:00, 12.51s/it]

Nulls: 0
'submission.csv' created.


,ID,PXD,Raw Data File,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],...,FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,,PXD061090,,,,,,,,,...,,,,,normal,,,,,
1,,PXD061285,,,,,,,,,...,,,,,cancer,,,,,
2,,PXD019519,,,,,,,,,...,,,,,cancer,,,,,
3,,PXD064564,,,,,,,,,...,,,,,cancer,,,,,
4,,PXD061195,,,,,,,,,...,,,,,normal,,,,,
